---

## NOTEBOOK 03 : DBSCAN - CLASSIFICATION AVEC DBSCAN

---

---

## Description

    Ce notebook constitue le livrable de la phase de modélisation du projet.
    Il est consacré à l'application de l'algorithme DBSCAN pour la classification des
    donnees et la détection des potentielles anomalies dans les transactions bancaires.

    OBJECTIFS:
    ----------
        - Entraîner un modèle DBSCAN sur les données nettoyées
        - Détecter les transactions potentiellement anormales
        - Analyser et visualiser les résultats
        - Interpréter les potentielles anomalies détectées
        - Sauvegarder le modèle pour une utilisation ultérieure

    CONTENU:
    --------
        1. Introduction théorique à DBSCAN
        2. Chargement et préparation des données
        3. Normalisation des variables
        4. Entraînement du modèle DBSCAN
        5. Évaluation et analyse des résultats
        6. Visualisation des Potentielles anomalies
        7. Export des Potentielles anomalies détectées
        8. Synthèse et recommandations

    PRÉREQUIS:
    ----------
        - Notebook 01 exécuté (données nettoyées disponibles)
        - Bibliothèques: pandas, numpy, matplotlib, seaborn, scikit-learn

---

### SECTION 1 : INTRODUCTION - RAPPEL THÉORIQUE

---

   #### 1.1. PRINCIPE DE L'ALGORITHME DBSCAN
   -----------------------------------------------
   DBSCAN (Density-Based Spatial Clustering of Applications with Noise) est un algorithme de
    clustering basé sur la densité des points.DBSCAN ne nécessite pas de
    spécifier le nombre de clusters à l’avance; à la place, il nécessite deux hyperparamètres propres :
    ε(epsilon) et minPts.

   FONDEMENT THÉORIQUE :

      - Les anomalies sont rares et différentes
      - La classification des points basee sur dansité peut nous aider et isole les arnomalies potentielles
      - L'algorithme construit des groupes ou clusters
      - Les anomalies ont la particulaté de ne pas etre dans aucun groupe

   #### 1.2. AVANTAGES DE L'APPROCHE
   -----------------------------
      - Non supervisé: ne nécessite pas de données étiquetées
      - Pas besoin de spécifier le nombre de clusters à l'avance
      - Détecte les formes de clusters arbitraires:pas besoin d’être sphériques
      - Gère bien le bruit et les outliers : robuste aux valeurs aberrantes  
      - Relativement efficace :Sa complexité moyenne est O(n log n) ou O(n2)

   #### 1.3. HYPERPARAMÈTRES CLÉS
   --------------------------
      ε(epsilon) : une distance rayonnant autour d’un point, définissant son voisinage immédiat (ε
        neighborhood)
      minPts : le nombre minimum de points requis dans le voisinage ε pour que l’on considère
        qu’un point est dans une région suffisamment dense.
      

   #### 1.4. INTERPRÉTATION DES RÉSULTATS
   ----------------------------------
      - Score = -1: Anomalie détectée
      - Score = 1: Transaction normale

---         

### Importation des bibliothèques

In [1]:
"""
2. IMPORTATION DES BIBLIOTHÈQUES
=================================

Cette section importe toutes les bibliothèques nécessaires pour:
    - La manipulation des données (pandas, numpy)
    - La visualisation (matplotlib, seaborn)
    - La modélisation (scikit-learn)
    - La gestion des fichiers (joblib, pathlib)
    - La gestion des avertissements (warnings)
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import joblib
import yaml
import sklearn
from datetime import datetime
from pathlib import Path
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
from prince import FAMD

# Configuration des warnings
warnings.filterwarnings('ignore')

# Configuration des graphiques
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12

# Création des dossiers nécessaires
for folder in ['../models', '../results/figures', '../results/metrics']:
    Path(folder).mkdir(parents=True, exist_ok=True)

print("="*60)
print("NOTEBOOK 03 : DBSCAN - CLASSIFICATION")
print("="*60)

print("\nBibliothèques importées avec succès")
print(f"   - Pandas: {pd.__version__}")
print(f"   - NumPy: {np.__version__}")
print(f"   - Matplotlib: {plt.matplotlib.__version__}")
print(f"   - Seaborn: {sns.__version__}")
print(f"   - Scikit-learn: {sklearn.__version__}")  
print(f"   - Joblib: {joblib.__version__}")


NOTEBOOK 03 : DBSCAN - CLASSIFICATION

Bibliothèques importées avec succès
   - Pandas: 2.3.3
   - NumPy: 2.2.5
   - Matplotlib: 3.10.7
   - Seaborn: 0.13.2
   - Scikit-learn: 1.7.2
   - Joblib: 1.5.3


In [2]:
"""
    Cette section charge le jeu de données nettoyé produit par le Notebook 01.
    Les données sont prêtes pour la modélisation.
"""

def load_cleaned_data(filepath: str) -> pd.DataFrame:
    """
    Charge le jeu de données nettoyé à partir d'un fichier CSV.
    
    Parameters
    ----------
    filepath : str
        Chemin vers le fichier CSV contenant les données nettoyées
        
    Returns
    -------
    pd.DataFrame
        DataFrame contenant les données nettoyées
        
    Raises
    ------
    FileNotFoundError
        Si le fichier spécifié n'existe pas
        
    Examples
    --------
    >>> df = load_cleaned_data('../data/processed/indian_banking_transactions_clean.csv')
    >>> print(df.shape)
    (550000, 20)
    
    Notes
    -----
    Cette fonction suppose que les données ont déjà été nettoyées
    et sont prêtes pour la modélisation.
    """
    
    try:
        df = pd.read_csv(filepath)
        print(f"Données chargées avec succès")
        print(f"   - Fichier: {filepath}")
        print(f"   - Dimensions: {df.shape[0]:,} lignes x {df.shape[1]} colonnes")
        return df
    except FileNotFoundError:
        print(f"Erreur: Fichier non trouvé à {filepath}")
        raise
    except Exception as e:
        print(f"Erreur lors du chargement: {str(e)}")
        raise

# Chargement des données
DATA_PATH = '../data/processed/indian_banking_transactions_clean.csv'

try:
    df = load_cleaned_data(DATA_PATH)
    print("\nAperçu des 5 premières lignes:")
    display(df.head())
except Exception as e:
    print(f"Erreur de chargement: {e}")
    df = pd.DataFrame()


Données chargées avec succès
   - Fichier: ../data/processed/indian_banking_transactions_clean.csv
   - Dimensions: 550,000 lignes x 20 colonnes

Aperçu des 5 premières lignes:


,transaction_id,customer_id,transaction_date,transaction_time,account_type,transaction_type,transaction_amount,transaction_direction,account_balance,merchant_category,state,credit_score,has_loan,loan_type,emi_amount,transaction_status,channel,kyc_status,is_fraud,transaction_hour
0,TXN000000001,CUST015796,2019-01-01,15:28:00,Current,UPI,1820.17,Debit,609365.31,Food & Dining,Maharashtra,764,0,Unknown,0.00,Success,Branch,Verified,0,15
1,TXN000000002,CUST000861,2019-01-01,03:00:00,Current,UPI,392.67,Credit,14451.14,Education,West Bengal,630,0,Unknown,0.00,Success,Mobile_App,Verified,0,3
2,TXN000000003,CUST076821,2019-01-01,18:03:00,Fixed Deposit,POS,1255.97,Debit,47621.87,Utilities,Punjab,813,1,Home,1433.91,Success,Mobile_App,Verified,0,18
3,TXN000000004,CUST054887,2019-01-01,08:03:00,Savings,UPI,2580.68,Debit,34467.85,Travel,Karnataka,628,1,Personal,6280.35,Success,API,Verified,0,8
4,TXN000000005,CUST006266,2019-01-01,14:23:00,Fixed Deposit,UPI,2573.80,Debit,26617.40,Utilities,West Bengal,767,0,Unknown,0.00,Success,API,Verified,0,14


### SECTION 4 : SÉLECTION DES VARIABLES POUR LA MODÉLISATION

#### SECTION 4.1 : SÉLECTION DES VARIABLES NUMÉRIQUES ET CATEGORIELLES

In [3]:
"""
Cette section sélectionne les variables numériques qui seront utilisées
pour la détection d'anomalies.

VARIABLES SÉLECTIONNÉES:
    - transaction_amount: Montant de la transaction
    - account_balance: Solde du compte
    - credit_score: Score de crédit du client
    - has_loan: Présence d'un prêt (0/1)
    - emi_amount: Montant des mensualités
    - transaction_hour: Heure de la transaction

JUSTIFICATION DES CHOIX:
    - Variables quantitatives continues ou discrètes
    - Variables pertinentes pour la détection d'anomalies
    - Absence de valeurs manquantes significatives
    - Indépendance relative (corrélations faibles)
"""

def select_features(df: pd.DataFrame) -> tuple:
    """
    Sélectionne les variables numériques pour la modélisation.
    
    Parameters
    ----------
    df : pd.DataFrame
        DataFrame contenant les données nettoyées
        
    Returns
    -------
    tuple
        - feature_names: Liste des noms des variables
        - categorical_features: Liste des noms des variables catégorielles
        
    Examples
    --------
    >>> numeric_features, categorical_features = select_features(df)
    >>> print(numeric_features)
    ['transaction_amount', 'account_balance', ...]
    """

    numeric_features = [
        'transaction_amount',
        'account_balance',
        'credit_score',
        'has_loan',
        'emi_amount',
        'transaction_hour'
    ]
    
    categorical_features = [
        'channel',
        'account_type',
        'transaction_type',
        'transaction_direction'
    ]
    
    # Vérifier la présence des colonnes
    all_features = numeric_features + categorical_features
    missing_cols = [col for col in all_features if col not in df.columns]
    if missing_cols:
        print(f"Colonnes manquantes: {missing_cols}")
        raise ValueError(f"Colonnes manquantes: {missing_cols}")
    
    print(f"\nMatrice de données:")
    print(f"   - Lignes: {df.shape[0]:,}")
    print(f"   - Variables numériques: {len(numeric_features)}")
    print(f"   - Variables catégorielles: {len(categorical_features)}")
    
    print("\nVariables numériques sélectionnées:")
    for i, feat in enumerate(numeric_features, 1):
        print(f"   {i}. {feat}")
    
    print("\nVariables catégorielles sélectionnées:")
    for i, feat in enumerate(categorical_features, 1):
        print(f"   {i}. {feat}")
    
    return numeric_features, categorical_features

numeric_features, categorical_features = select_features(df)


Matrice de données:
   - Lignes: 550,000
   - Variables numériques: 6
   - Variables catégorielles: 4

Variables numériques sélectionnées:
   1. transaction_amount
   2. account_balance
   3. credit_score
   4. has_loan
   5. emi_amount
   6. transaction_hour

Variables catégorielles sélectionnées:
   1. channel
   2. account_type
   3. transaction_type
   4. transaction_direction


#### SECTION 4.2 : ENCODAGE DES VARIABLES CATÉGORIELLES

In [4]:
"""
Cette section encode les variables catégorielles pour les rendre utilisables
par les algorithmes de machine learning.

VARIABLES CATÉGORIELLES ENCODÉES:
    - channel: Canal de transaction (Mobile_App, Web, ATM, etc.)
    - account_type: Type de compte (Savings, Current, Salary, etc.)
    - transaction_type: Type de transaction (UPI, NEFT, POS, etc.)
    - transaction_direction: Sens du flux (Debit, Credit)

MÉTHODE UTILISÉE: One-Hot Encoding
    - Transforme chaque catégorie en une colonne binaire (0 ou 1)
    - Évite de créer une relation d'ordre entre les catégories

JUSTIFICATION:
    - Les variables catégorielles apportent un contexte métier essentiel
    - Elles permettent au modèle de distinguer les profils de risque
    - Exemple: Une fraude sur Mobile_App n'a pas le même profil qu'une fraude sur Branch
"""

def encode_categorical_features(df: pd.DataFrame, numeric_features: list, categorical_features: list) -> tuple:
    """
    Encode les variables catégorielles avec One-Hot Encoding.
    
    Parameters
    ----------
    df : pd.DataFrame
        DataFrame contenant les données nettoyées
    numeric_features : list
        Liste des noms des variables numériques
    categorical_features : list
        Liste des noms des variables catégorielles
        
    Returns
    -------
    tuple
        - X_encoded: Matrice numpy avec les variables encodées
        - feature_names: Liste des noms de toutes les variables
        - encoder: Objet OneHotEncoder ajusté
        
    Examples
    --------
    >>> X_encoded, feature_names, encoder = encode_categorical_features(df, numeric_features, categorical_features)
    >>> print(X_encoded.shape)
    (550000, 45)
    """
    
    print("\n" + "="*60)
    print("4.1. ENCODAGE DES VARIABLES CATÉGORIELLES")
    print("="*60)
    
    # Extraire les variables numériques
    X_numeric = df[numeric_features].values
    
    # Encoder les variables catégorielles
    print("\nApplication du One-Hot Encoding...")
    encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
    X_categorical = encoder.fit_transform(df[categorical_features])
    
    # Récupérer les noms des colonnes encodées
    categorical_names = encoder.get_feature_names_out(categorical_features)
    
    # Combiner les variables
    X_encoded = np.hstack([X_numeric, X_categorical])
    all_feature_names = numeric_features + list(categorical_names)
    
    print(f"\nEncodage terminé!")
    print(f"   - Matrice X: {X_encoded.shape[0]:,} lignes x {X_encoded.shape[1]} colonnes")
    print(f"   - Variables numériques: {len(numeric_features)}")
    print(f"   - Variables catégorielles encodées: {len(categorical_names)}")
    print(f"   - Total variables: {X_encoded.shape[1]}")
    
    # Afficher quelques noms de colonnes encodées
    print(f"\nExemples de colonnes encodées:")
    for i, name in enumerate(categorical_names[:10], 1):
        print(f"      {i}. {name}")
    if len(categorical_names) > 10:
        print(f"      ... et {len(categorical_names) - 10} autres")
    
    # Sauvegarder l'encodeur
    joblib.dump(encoder, '../models/onehot_encoder.pkl')
    print("\nEncodeur sauvegardé: '../models/onehot_encoder.pkl'")
    
    return X_encoded, all_feature_names, encoder

X_encoded, feature_names, encoder = encode_categorical_features(df, numeric_features, categorical_features)


4.1. ENCODAGE DES VARIABLES CATÉGORIELLES

Application du One-Hot Encoding...

Encodage terminé!
   - Matrice X: 550,000 lignes x 29 colonnes
   - Variables numériques: 6
   - Variables catégorielles encodées: 23
   - Total variables: 29

Exemples de colonnes encodées:
      1. channel_API
      2. channel_ATM
      3. channel_Branch
      4. channel_Mobile_App
      5. channel_POS_Terminal
      6. channel_Web
      7. account_type_Current
      8. account_type_Fixed Deposit
      9. account_type_NRI
      10. account_type_Salary
      ... et 13 autres

Encodeur sauvegardé: '../models/onehot_encoder.pkl'


### SECTION 5 : NORMALISATION DES DONNÉES (STANDARD SCALER)

In [5]:
"""
Cette section applique une transformation logarithmique pour réduire
l'asymétrie des données avant la normalisation.

RAISON DE LA LOG-TRANSFORMATION:
    - Réduire l'asymétrie des distributions (skewness)
    - Stabiliser la variance
    - Réduire l'impact des valeurs extrêmes (outliers)
    - Améliorer la performance des algorithmes

VARIABLES TRANSFORMÉES:
    - transaction_amount: log(1 + amount)
    - account_balance: log(1 + balance)
    - emi_amount: log(1 + emi) (si présente)

MÉTHODE: np.log1p(x) = log(1 + x)
    - Évite les problèmes avec log(0)
    - Préserve l'ordre des valeurs
"""

def normalize_data(X: np.ndarray, numeric_features: list, df: pd.DataFrame) -> tuple:
    """
    Applique une transformation logarithmique sur les montants/soldes 
    puis normalise les données en utilisant StandardScaler.
    
    Parameters
    ----------
    X : np.ndarray
        Matrice des données encodées
    numeric_features : list
        Liste des noms des variables numériques
    df : pd.DataFrame
        DataFrame original
        
    Returns
    -------
    tuple
        - X_scaled: Array numpy des données transformées et normalisées
        - scaler: Objet StandardScaler ajusté
        
    Notes
    -----
    La transformation log1p est appliquée sur:
        - transaction_amount
        - account_balance
        - emi_amount (si présente)
        
    Examples
    --------
    >>> X_scaled, scaler = normalize_data(X_encoded, numeric_features, df)
    >>> print(X_scaled.mean())
    0.0
    """
    
    print("\n" + "="*60)
    print("5. TRANSFORMATION LOG + NORMALISATION")
    print("="*60)
    
    # Copie pour éviter de modifier l'original
    X_transformed = X.copy()
    
    # Variables à transformer par log1p
    log_vars = ['transaction_amount', 'account_balance']
    if 'emi_amount' in df.columns:
        log_vars.append('emi_amount')
    
    # Trouver les indices des colonnes à transformer
    log_indices = []
    for i, feat in enumerate(numeric_features):
        if feat in log_vars:
            log_indices.append(i)
    
    print("\nVariables transformées par log1p:")
    for feat in log_vars:
        print(f"   - {feat}")
    
    # Application de la transformation log1p
    for idx in log_indices:
        X_transformed[:, idx] = np.log1p(X_transformed[:, idx])
    
    # Statistiques avant normalisation
    print("\nStatistiques après transformation log1p:")
    for idx, feat in zip(log_indices, log_vars):
        print(f"   - {feat}: min={X_transformed[:, idx].min():.4f}, "
              f"max={X_transformed[:, idx].max():.4f}, "
              f"mean={X_transformed[:, idx].mean():.4f}")
    
    # Normalisation StandardScaler
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_transformed)
    
    print("\nStatistiques après StandardScaler:")
    print(f"   - Moyenne globale: {X_scaled.mean():.6f}")
    print(f"   - Écart-type global: {X_scaled.std():.6f}")
    
    # Sauvegarder le scaler
    joblib.dump(scaler, '../models/dbscan_scaler.pkl')
    print("\nScaler sauvegardé: '../models/dbscan_scaler.pkl'")
    
    return X_scaled, scaler

X_scaled, scaler = normalize_data(X_encoded, numeric_features, df)


5. TRANSFORMATION LOG + NORMALISATION

Variables transformées par log1p:
   - transaction_amount
   - account_balance
   - emi_amount

Statistiques après transformation log1p:
   - transaction_amount: min=1.2238, max=16.1181, mean=7.9744
   - account_balance: min=6.2166, max=15.4249, mean=10.5024
   - emi_amount: min=0.0000, max=11.9403, mean=2.9717

Statistiques après StandardScaler:
   - Moyenne globale: 0.000000
   - Écart-type global: 1.000000

Scaler sauvegardé: '../models/dbscan_scaler.pkl'


### SECTION 6: REDUCTION DE DIMENSIONNALITE-FAMD

In [7]:
"""
Cette section applique la FAMD avant DBSCAN.
DBSCAN repose sur la distance entre transactions. Cette méthode gère mieux
les variables numériques et catégorielles mixtes qu'une simple ACP sur
un encodage one-hot.
"""

def reduce_dimensions_famd(
    df: pd.DataFrame,
    numeric_features: list,
    categorical_features: list,
    config_path: str = '../configs/config.yml'
) -> tuple:

    """
    Projette les données mixtes dans l'espace FAMD.

    Parameters
    ----------
    df : pd.DataFrame
        DataFrame contenant les variables numériques et catégorielles
    numeric_features : list
        Liste des variables numériques
    categorical_features : list
        Liste des variables catégorielles
    config_path : str
        Chemin vers le fichier de configuration

    Returns
    -------
    tuple
        - X_reduced : données projetées pour DBSCAN
        - famd : objet FAMD ajusté
    """

    n_components = 10
    try:
        with open(config_path, 'r', encoding='utf-8') as file:
            famd_config = (yaml.safe_load(file) or {}).get('famd', {})
        n_components = famd_config.get('n_components', n_components)
    except FileNotFoundError:
        print('Configuration FAMD introuvable : utilisation de 10 composantes.')

    if not isinstance(n_components, int) or n_components < 2:
        raise ValueError('famd.n_components doit être un entier supérieur ou égal à 2.')

    print('\n' + '=' * 60)
    print('6. RÉDUCTION DE DIMENSIONNALITÉ PAR FAMD')
    print('=' * 60)

    famd = FAMD(n_components=n_components, random_state=42)
    X_reduced = famd.fit_transform(df[numeric_features + categorical_features])

    explained_inertia = getattr(famd, 'explained_inertia_', None)
    if explained_inertia is not None:
        print(f'Composantes retenues : {n_components}')
        print(f'Inertie expliquée cumulée : {np.sum(explained_inertia) * 100:.2f}%')
    else:
        print(f'Composantes retenues : {n_components}')

    print(f'Nouvelle matrice : {X_reduced.shape[0]:,} lignes x {X_reduced.shape[1]} composantes')

    joblib.dump(famd, '../models/famd_dbscan.pkl')
    print("FAMD sauvegardée : '../models/famd_dbscan.pkl'")
    return X_reduced, famd


X_reduced, famd_model = reduce_dimensions_famd(df, numeric_features, categorical_features)



6. RÉDUCTION DE DIMENSIONNALITÉ PAR FAMD


MemoryError: Unable to allocate 4.20 MiB for an array with shape (550000,) and data type float64

#### 6.1. Visualisation de la FAMD

In [ ]:
"""
Dans cette section, pour justifier le nombre de composantes retenues,
on visualise la projection FAMD en 2D et, si disponible, la variance expliquée.
"""
def visualize_famd_results(
    X_reduced: np.ndarray,
    famd_model,
    sample_size: int = 550000,
    random_state: int = 42
) -> None:

    """
    Affiche la projection FAMD en 2D et la variance expliquée si disponible.
    """
    explained_inertia = getattr(famd_model, 'explained_inertia_', None)
    if explained_inertia is not None:
        cumulative_variance = np.cumsum(explained_inertia)
        component_numbers = np.arange(1, len(cumulative_variance) + 1)
        variance_threshold = cumulative_variance[-1]
        n_components_threshold = len(explained_inertia)
    else:
        cumulative_variance = None
        component_numbers = None
        variance_threshold = None
        n_components_threshold = X_reduced.shape[1]

    if X_reduced.shape[1] >= 2:
        projection_2d = X_reduced[:, :2]
        if explained_inertia is not None:
            pc1_variance, pc2_variance = explained_inertia[:2]
        else:
            pc1_variance, pc2_variance = 0.0, 0.0
    else:
        raise ValueError('La visualisation 2D nécessite au moins deux composantes FAMD.')

    if len(projection_2d) > sample_size:
        rng = np.random.default_rng(random_state)
        sample_index = rng.choice(len(projection_2d), size=sample_size, replace=False)
        projection_plot = projection_2d[sample_index]
        sample_note = f'Echantillon affiché : {sample_size:,} / {len(projection_2d):,} transactions'
    else:
        projection_plot = projection_2d
        sample_note = f'Transactions affichées : {len(projection_2d):,}'

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    if cumulative_variance is not None:
        axes[0].plot(component_numbers, cumulative_variance, marker='o', linewidth=2)
        axes[0].set_title('Inertie expliquée cumulée par la FAMD')
        axes[0].set_xlabel('Nombre de composantes')
        axes[0].set_ylabel('Inertie expliquée cumulée')
        axes[0].set_xticks(component_numbers)
        axes[0].set_ylim(0, 1.05)
        y_ticks = np.linspace(0, 1, 6)
        axes[0].set_yticks(y_ticks)
        axes[0].set_yticklabels([f'{value:.0%}' for value in y_ticks])
        axes[0].legend([f'Total: {variance_threshold:.0%}'], loc='lower right')
    else:
        axes[0].text(0.5, 0.5, 'Inertie expliquée non disponible pour FAMD',
                     ha='center', va='center', fontsize=12)
        axes[0].axis('off')

    axes[1].scatter(
        projection_plot[:, 0],
        projection_plot[:, 1],
        s=8,
        alpha=0.35,
        color='#1f77b4',
        edgecolors='none'
    )
    axes[1].set_title('Projection FAMD en 2D')
    axes[1].set_xlabel(f'Dimension 1 ({pc1_variance:.1%} de variance/inertie)')
    axes[1].set_ylabel(f'Dimension 2 ({pc2_variance:.1%} de variance/inertie)')
    axes[1].text(
        0.02, 0.98, sample_note,
        transform=axes[1].transAxes,
        va='top',
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.8, edgecolor='lightgray')
    )

    plt.tight_layout()
    plt.savefig('../results/figures/famd_projection_2d.png', dpi=300, bbox_inches='tight')
    plt.show()
    print("Visualisation FAMD sauvegardée : '../results/figures/famd_projection_2d.png'")


visualize_famd_results(X_reduced, famd_model)


### SECTION 7 : CONFIGURATION DU MODÈLE DBSCAN

In [ ]:
"""
    
    Cette section charge et complète la configuration DBSCAN.
    Les hyperparamètres eps et min_samples ne sont pas fixés a priori, ils sont
    calibrés ci-dessous par les courbes des k-distances sur toutes les données FAMD.

"""

def load_dbscan_config(config_path: str = '../configs/config.yml') -> tuple:

    """
        Charge les paramètres DBSCAN avec des valeurs par défaut sûres.

        retournes:
            tuple: config , dbscan_config

    """

    defaults = {
        'metric': 'euclidean',
        'min_samples_candidates': [5, 10, 15, 20]
    }

    try:
        with open(config_path, 'r', encoding='utf-8') as file:
            config = yaml.safe_load(file) or {}
        print('Configuration chargée avec succès')
    except FileNotFoundError:
        print(f'Fichier de configuration non trouvé : {config_path}')
        print('Utilisation des paramètres par défaut')
        config = {}

    dbscan_config = {**defaults, **config.get('dbscan', {})}
    candidates = dbscan_config['min_samples_candidates']
    if not all(isinstance(value, int) and value >= 2 for value in candidates):
        raise ValueError('Chaque candidat min_samples doit être un entier supérieur ou égal à 2.')
    print(f"Candidats min_samples chargés : {candidates}")

    return config, dbscan_config


def _find_knee(sorted_distances: np.ndarray) -> tuple:

    """
        Détecte le coude maximal par sa distance à la droite des extrémités.

        retournes:
            tuple: knee_index, distances_to_line
    """
    x = np.linspace(0, 1, len(sorted_distances))
    y_range = sorted_distances[-1] - sorted_distances[0]
    y = (sorted_distances - sorted_distances[0]) / y_range if y_range else np.zeros_like(x)
    # La distance a cette droite est maximale au changement de pente (le coude).
    distances_to_line = np.abs(y - x) / np.sqrt(2)
    knee_index = int(np.argmax(distances_to_line))

    return knee_index, float(distances_to_line[knee_index])


def calibrate_dbscan_kdistances(X_reduced: np.ndarray, config: dict) -> dict:
    
    """ 
        Cette fonction  choisit min_samples et eps à partir des k-distances calculées sur toutes les lignes.
        Pour chaque valeur candidate de min_samples, la distance au k-ième voisin est triée.
        La valeur candidate dont la courbe présente le coude le plus net est retenu, eps est
        alors égal à la k-distance située à ce coude.

        retourne:
            dict: config
    """

    print('\n' + '=' * 60)
    print('7. CALIBRATION DBSCAN PAR LES K-DISTANCES')
    print('=' * 60)

    candidates = sorted(set(config['min_samples_candidates']))
    candidates = [value for value in candidates if value <= len(X_reduced)]
    if not candidates:
        raise ValueError('Aucun candidat min_samples n’est compatible avec la taille des données.')

    curves, results = {}, []
    for min_samples in candidates:
        print(f'Calcul des k-distances pour min_samples={min_samples}...')
        neighbors = NearestNeighbors(
            n_neighbors=min_samples, metric=config['metric'], algorithm='ball_tree', n_jobs=-1
        ).fit(X_reduced)
        distances, _ = neighbors.kneighbors(X_reduced)
        k_distances = np.sort(distances[:, -1])
        knee_index, knee_strength = _find_knee(k_distances)
        curves[min_samples] = (k_distances, knee_index)
        results.append({
            'min_samples': min_samples,
            'eps': float(k_distances[knee_index]),
            'knee_strength': knee_strength
        })

    best = max(results, key=lambda result: result['knee_strength'])
    config['min_samples'] = best['min_samples']
    config['eps'] = best['eps']

    # graphique
    fig, ax = plt.subplots(figsize=(12, 6))
    _, best_knee_index = curves[config['min_samples']]
    for min_samples, (k_distances, _) in curves.items():
        ax.plot(k_distances, linewidth=1, label=f'min_samples={min_samples}')
    ax.axhline(config['eps'], color='red', linestyle='--',
               label=f"eps retenu = {config['eps']:.4f}")
    ax.scatter(best_knee_index, config['eps'], color='red', zorder=3)
    ax.set_title('Courbes des k-distances pour la calibration de DBSCAN')
    ax.set_xlabel('Transactions triées')
    ax.set_ylabel('Distance au k-ième voisin')
    ax.legend()
    plt.tight_layout()
    plt.savefig('../results/figures/dbscan_k_distances.png', dpi=300, bbox_inches='tight')
    plt.show()

    print(f"\nParamètres retenus : min_samples={config['min_samples']}, eps={config['eps']:.4f}")
    print("Graphique sauvegardé : '../results/figures/dbscan_k_distances.png'")

    return config


config, dbscan_config = load_dbscan_config()
dbscan_config = calibrate_dbscan_kdistances(X_reduced, dbscan_config)

print('\nParamètres du modèle DBSCAN :')
for key, value in dbscan_config.items():
    print(f'   - {key}: {value}')


### SECTION 8: ENTRAÎNEMENT DU MODÈLE

In [ ]:


def train_dbscan(X_reduced: np.ndarray, config: dict) -> tuple:

    """
        Cette fonction entraîne DBSCAN sur les données déjà réduites par la FAMD.
        Le label -1 est le bruit selon DBSCAN, il est interprété ici
        comme une anomalie potentielle. Les autres labels correspondent à des groupes
        denses. 

        retournes:
          tuple: model, labels

    """
    print('\n' + '=' * 60)
    print('8. ENTRAÎNEMENT DU MODÈLE DBSCAN')
    print('=' * 60)

    print(f'Jeu complet utilisé pour DBSCAN : {len(X_reduced):,} transactions')

    # on passe les parametres à DBSCANB
    model = DBSCAN(
        eps=config['eps'],
        min_samples=config['min_samples'],
        metric=config['metric'],
        algorithm='ball_tree',
        n_jobs=-1
    )

    # on entraine le model
    print('\nEntraînement du modèle...')
    labels = model.fit_predict(X_reduced)

    n_anomalies = int(np.sum(labels == -1))
    n_clusters = len(set(labels)) - int(-1 in labels)
    pct_anomalies = n_anomalies / len(labels) * 100
    print('\n Résultats de la détection :')
    print(f'   Clusters denses détectés : {n_clusters:,}')
    print(f'   Transactions normales : {len(labels) - n_anomalies:,}')
    print(f'   Anomalies potentielles (bruit, label -1) : {n_anomalies:,} ({pct_anomalies:.2f}%)')

    # Le bundle conserve la FAMD et les paramètres calibrés pour assurer la traçabilité.
    artifact = {
        'dbscan_model': model,
        'famd': famd_model,
        'config': config,
        'feature_names': feature_names
    }
    joblib.dump(artifact, '../models/dbscan_famd.pkl')
    print("\nModèle et FAMD sauvegardés : '../models/dbscan_famd.pkl'")

    return model, labels


dbscan_model, dbscan_labels = train_dbscan(
    X_reduced, dbscan_config
)


### SECTION 9 : VISUALISATION DES CLUSTERS DBSCAN

In [ ]:
"""
Cette visualisation projette les transactions sur les deux premières composantes FAMD
et colore les points selon le cluster attribué par DBSCAN. Le label -1 correspond au
bruit, c'est-à-dire aux anomalies potentielles.
"""
def visualize_dbscan_clusters_2d(
    X_reduced: np.ndarray,
    labels: np.ndarray,
    famd_model,
    sample_size: int = 550000,
    random_state: int = 42
) -> None:
    """
    Affiche les clusters DBSCAN dans le plan FAMD.
    """

    labels = np.asarray(labels)
    if len(X_reduced) != len(labels):
        raise ValueError('X_reduced et labels doivent avoir le même nombre de lignes.')
    if X_reduced.shape[1] < 2:
        raise ValueError('La visualisation 2D nécessite au moins deux composantes FAMD.')

    projection_2d = X_reduced[:, :2]
    explained_inertia = getattr(famd_model, 'explained_inertia_', None)
    if explained_inertia is not None and len(explained_inertia) >= 2:
        dim1_variance, dim2_variance = explained_inertia[:2]
    else:
        dim1_variance, dim2_variance = 0.0, 0.0

    if len(labels) > sample_size:
        rng = np.random.default_rng(random_state)
        noise_index = np.flatnonzero(labels == -1)
        cluster_index = np.flatnonzero(labels != -1)

        if len(noise_index) >= sample_size:
            sample_index = rng.choice(noise_index, size=sample_size, replace=False)
        else:
            remaining = sample_size - len(noise_index)
            cluster_sample = rng.choice(cluster_index, size=min(remaining, len(cluster_index)), replace=False)
            sample_index = np.concatenate([noise_index, cluster_sample])
        sample_note = f'Echantillon affiché : {len(sample_index):,} / {len(labels):,} transactions'
    else:
        sample_index = np.arange(len(labels))
        sample_note = f'Transactions affichées : {len(labels):,}'

    projection_plot = projection_2d[sample_index]
    labels_plot = labels[sample_index]
    cluster_labels = sorted(label for label in np.unique(labels_plot) if label != -1)
    cluster_sizes = {label: int(np.sum(labels_plot == label)) for label in cluster_labels}
    clusters_by_size_desc = sorted(cluster_labels, key=lambda label: cluster_sizes[label], reverse=True)

    fig, ax = plt.subplots(figsize=(12, 8))
    cluster_colors = {
        0: '#0072B2',
        1: '#E69F00',
    }
    fallback_palette = sns.color_palette('Dark2', n_colors=max(len(cluster_labels), 1))

    for color_index, cluster_label in enumerate(clusters_by_size_desc):
        mask = labels_plot == cluster_label
        color = cluster_colors.get(cluster_label, fallback_palette[color_index % len(fallback_palette)])
        ax.scatter(
            projection_plot[mask, 0],
            projection_plot[mask, 1],
            s=10,
            alpha=0.55,
            color=color,
            edgecolors='none',
            label=f'Cluster {cluster_label} ({cluster_sizes[cluster_label]:,})'
        )

    noise_mask = labels_plot == -1
    if np.any(noise_mask):
        ax.scatter(
            projection_plot[noise_mask, 0],
            projection_plot[noise_mask, 1],
            s=14,
            alpha=0.75,
            color='#d62728',
            edgecolors='none',
            label=f'Bruit (-1) ({int(np.sum(noise_mask)):,})'
        )

    ax.set_title('Arrangement des clusters DBSCAN dans le plan FAMD')
    ax.set_xlabel(f'Dimension 1 ({dim1_variance:.1%} de l\'inertie)')
    ax.set_ylabel(f'Dimension 2 ({dim2_variance:.1%} de l\'inertie)')
    ax.text(
        0.02, 0.98, sample_note,
        transform=ax.transAxes,
        va='top',
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.8, edgecolor='lightgray')
    )
    ax.legend(loc='best', markerscale=2)

    plt.tight_layout()
    output_path = '../results/figures/dbscan_clusters_famd_2d.png'
    plt.savefig(output_path, dpi=300, bbox_inches='tight')
    plt.close(fig)
    print(f"Visualisation des clusters sauvegardée : '{output_path}'")


visualize_dbscan_clusters_2d(X_reduced, dbscan_labels, famd_model)


### SECTION 10 : ANALYSE DES RÉSULTATS